# Chapitre 6 : réfléchir avant de répondre (raisonnement pas à pas)

- [Cours](#cours)
- [Exercices](#exercices)
- [Espace de test](#espace-de-test)

## Configuration

Exécutez la cellule de configuration suivante pour charger votre clé API et définir la fonction utilitaire `get_completion`.

In [ ]:
!pip install anthropic

# Importer la bibliothèque standard de Python pour les expressions régulières
import re
import anthropic

# Récupérer les variables API_KEY et MODEL_NAME depuis le store IPython
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt="", prefill=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ]
    )
    return message.content[0].text

---

## Cours

Si quelqu'un vous réveillait et vous posait aussitôt plusieurs questions compliquées exigeant une réponse immédiate, comment vous en sortiriez-vous ? Sans doute moins bien que si vous aviez le temps de **réfléchir d'abord à votre réponse**.

Il en va de même pour Claude.

**Laisser à Claude le temps de réfléchir étape par étape le rend parfois plus précis**, en particulier sur les tâches complexes. Mais **la réflexion ne compte que si elle est écrite**. Vous ne pouvez pas demander à Claude de réfléchir tout en n'affichant que la réponse : dans ce cas, aucune réflexion n'a réellement eu lieu.

### Exemples

Dans le prompt ci-dessous, un lecteur humain voit bien que la deuxième phrase contredit la première. Mais **Claude prend le mot « unrelated » (sans rapport) trop au pied de la lettre**.

In [ ]:
# Prompt
PROMPT = """Is this movie review sentiment positive or negative?

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since the year 1900."""

# Afficher la réponse de Claude
print(get_completion(PROMPT))

Pour améliorer la réponse, **laissons Claude réfléchir avant de répondre**. Pour cela, nous détaillons littéralement les étapes qu'il doit suivre pour traiter la tâche. Avec un soupçon de role prompting, Claude comprend l'avis beaucoup plus finement.

In [ ]:
# Prompt système
SYSTEM_PROMPT = "You are a savvy reader of movie reviews."

# Prompt
PROMPT = """Is this review sentiment positive or negative? First, write the best arguments for each side in <positive-argument> and <negative-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since 1900."""

# Afficher la réponse de Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))

**Claude est parfois sensible à l'ordre des éléments.** Cet exemple est à la limite de sa capacité à comprendre un texte nuancé : si l'on inverse l'ordre des arguments de l'exemple précédent, pour mettre le négatif en premier et le positif en second, son évaluation globale devient positive.

Dans la plupart des cas (mais pas tous, ce qui complique les choses), **Claude a tendance à choisir la seconde de deux options**, peut-être parce que, dans ses données d'entraînement issues du web, la seconde option était plus souvent la bonne.

In [ ]:
# Prompt
PROMPT = """Is this review sentiment negative or positive? First write the best arguments for each side in <negative-argument> and <positive-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. Unrelatedly, I have been living under a rock since 1900."""

# Afficher la réponse de Claude
print(get_completion(PROMPT))

**Laisser Claude réfléchir peut transformer une réponse fausse en réponse juste.** Dans bien des cas où Claude se trompe, c'est aussi simple que cela !

Prenons un exemple où la réponse de Claude est fausse, pour voir comment lui demander de réfléchir corrige le problème.

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956."

# Afficher la réponse de Claude
print(get_completion(PROMPT))

Corrigeons cela en demandant à Claude de réfléchir étape par étape, cette fois entre balises `<brainstorm>`.

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956. First brainstorm about some actors and their birth years in <brainstorm> tags, then give your answer."

# Afficher la réponse de Claude
print(get_completion(PROMPT))

Si vous voulez expérimenter avec les prompts du cours sans modifier le contenu ci-dessus, descendez tout en bas du notebook jusqu'à l'[**Espace de test**](#espace-de-test).

---

## Exercices
- [Exercice 6.1 - Classer des e-mails](#exercice-61---classer-des-e-mails)
- [Exercice 6.2 - Format de la classification](#exercice-62---format-de-la-classification)

### Exercice 6.1 - Classer des e-mails
Dans cet exercice, vous demandez à Claude de classer des e-mails dans les catégories suivantes :
- (A) Pre-sale question (question avant achat)
- (B) Broken or defective item (article cassé ou défectueux)
- (C) Billing question (question de facturation)
- (D) Other (please explain) (autre, à expliquer)

Pour la première partie de l'exercice, modifiez le `PROMPT` pour **que Claude donne la bonne catégorie, et UNIQUEMENT la catégorie**. La réponse doit **contenir la lettre (A à D) de la bonne catégorie, avec les parenthèses, ainsi que le nom de la catégorie**.

Les commentaires à côté de chaque e-mail de la liste `EMAILS` indiquent la catégorie attendue.

Note : le correcteur compare la réponse aux libellés anglais des catégories. Conservez-les tels quels dans votre prompt.

In [ ]:
# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = """Please classify this email as either green or blue: {email}"""

# Pré-remplissage de la réponse de Claude, le cas échéant
PREFILL = ""

# Contenu de la variable, stocké sous forme de liste
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???", # (C) Billing question
    "How did I get here I am not good with computer.  Halp." # (D) Other (please explain)
]

# Catégories correctes, stockées en liste de listes car un e-mail peut avoir plusieurs catégories correctes
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Dictionnaire des chaînes de chaque catégorie, utilisé pour la correction par regex
REGEX_CATEGORIES = {
    "A": "A\) P",
    "B": "B\) B",
    "C": "C\) B",
    "D": "D\) O"
}

# Parcourir la liste des e-mails
for i,email in enumerate(EMAILS):
    
    # Insérer le texte de l'e-mail dans la variable email
    formatted_prompt = PROMPT.format(email=email)
   
    # Obtenir la réponse de Claude
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Corriger la réponse de Claude
    grade = any([bool(re.search(REGEX_CATEGORIES[ans], response)) for ans in ANSWERS[i]])
    
    # Afficher la réponse de Claude
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print("\nASSISTANT TURN")
    print(PREFILL)
    print("\n------------------------------------- Claude's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", grade, "\n\n\n\n\n\n")

❓ Pour obtenir un indice, exécutez la cellule ci-dessous.

In [ ]:
from hints import exercise_6_1_hint; print(exercise_6_1_hint)

Toujours bloqué ? Exécutez la cellule ci-dessous pour voir un exemple de solution.

In [ ]:
from hints import exercise_6_1_solution; print(exercise_6_1_solution)

### Exercice 6.2 - Format de la classification
Dans cet exercice, nous affinons la sortie du prompt précédent pour obtenir une réponse formatée exactement comme nous le voulons.

Utilisez votre technique de formatage préférée pour que Claude place UNIQUEMENT la lettre de la bonne catégorie entre balises `<answer></answer>`. Par exemple, la réponse au premier e-mail doit contenir exactement la chaîne `<answer>B</answer>`.

Reportez-vous aux commentaires à côté de chaque e-mail de la liste `EMAILS` si vous ne savez plus quelle lettre correspond à quel e-mail.

In [ ]:
# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = """Please classify this email as either green or blue: {email}"""

# Pré-remplissage de la réponse de Claude, le cas échéant
PREFILL = ""

# Contenu de la variable, stocké sous forme de liste
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???", # (C) Billing question
    "How did I get here I am not good with computer.  Halp." # (D) Other (please explain)
]

# Catégories correctes, stockées en liste de listes car un e-mail peut avoir plusieurs catégories correctes
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Dictionnaire des chaînes de chaque catégorie, utilisé pour la correction par regex
REGEX_CATEGORIES = {
    "A": "<answer>A</answer>",
    "B": "<answer>B</answer>",
    "C": "<answer>C</answer>",
    "D": "<answer>D</answer>"
}

# Parcourir la liste des e-mails
for i,email in enumerate(EMAILS):
    
    # Insérer le texte de l'e-mail dans la variable email
    formatted_prompt = PROMPT.format(email=email)
   
    # Obtenir la réponse de Claude
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Corriger la réponse de Claude
    grade = any([bool(re.search(REGEX_CATEGORIES[ans], response)) for ans in ANSWERS[i]])
    
    # Afficher la réponse de Claude
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print("\nASSISTANT TURN")
    print(PREFILL)
    print("\n------------------------------------- Claude's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", grade, "\n\n\n\n\n\n")

❓ Pour obtenir un indice, exécutez la cellule ci-dessous.

In [ ]:
from hints import exercise_6_2_hint; print(exercise_6_2_hint)

### Bravo !

Si vous avez résolu tous les exercices jusqu'ici, vous pouvez passer au chapitre suivant. Bon prompting !

---

## Espace de test

Cet espace vous permet d'expérimenter librement avec les exemples de prompts de la leçon et de les modifier pour voir l'effet sur les réponses de Claude.

In [ ]:
# Prompt
PROMPT = """Is this movie review sentiment positive or negative?

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since the year 1900."""

# Afficher la réponse de Claude
print(get_completion(PROMPT))

In [ ]:
# Prompt système
SYSTEM_PROMPT = "You are a savvy reader of movie reviews."

# Prompt
PROMPT = """Is this review sentiment positive or negative? First, write the best arguments for each side in <positive-argument> and <negative-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since 1900."""

# Afficher la réponse de Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))

In [ ]:
# Prompt
PROMPT = """Is this review sentiment negative or positive? First write the best arguments for each side in <negative-argument> and <positive-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. Unrelatedly, I have been living under a rock since 1900."""

# Afficher la réponse de Claude
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956."

# Afficher la réponse de Claude
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956. First brainstorm about some actors and their birth years in <brainstorm> tags, then give your answer."

# Afficher la réponse de Claude
print(get_completion(PROMPT))